# P2 — DAG-inspired causal graph baseline — Colab runnable

**Compute:** L4 or A100 recommended; T4 works but RoBERTa-large feature caching is slower.

Upload `iemocap.pkl`; the notebook installs everything else.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil
try:
    from google.colab import files
    print('Upload iemocap.pkl')
    files.upload()
except Exception:
    pass
subprocess.check_call([sys.executable,'-m','pip','install','-q','torch','transformers==4.48.3','scikit-learn','pyyaml','sentencepiece'])
ROOT=Path.cwd()
cands=list(ROOT.glob('iemocap.pkl'))+list(ROOT.glob('IEMOCAP.pkl'))+list(Path('/content').glob('iemocap.pkl'))
if not cands: raise FileNotFoundError('Upload iemocap.pkl')
DATA_PATH=str(cands[0].resolve()); print(DATA_PATH)
subprocess.run(['nvidia-smi'],check=False)

In [ ]:
!python smoke_test.py
!python run.py --mode cache --data_path "$DATA_PATH" --config config.yaml --cache_dir outputs/cache
!python run.py --mode train --data_path "$DATA_PATH" --config config.yaml --cache_dir outputs/cache --output_dir outputs/full
!python run.py --mode eval --data_path "$DATA_PATH" --config config.yaml --cache_dir outputs/cache --model_path outputs/full/best_model.pt --split test --save_path outputs/full/predictions.json

In [ ]:
import shutil
print(shutil.make_archive('outputs_'+Path.cwd().name,'zip','outputs'))